In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os, random, pickle, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter
from PIL import Image

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.models as models

import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import train_test_split
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score)

SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

In [ ]:
CASIA_AU   = Path('/kaggle/input/datasets/chongtrung/casia-v2/CASIA2/Au')
CASIA_TP   = Path('/kaggle/input/datasets/chongtrung/casia-v2/CASIA2/Tp')
DEFACTO_CM = Path('/kaggle/input/datasets/defactodataset/defactocopymove/copymove_img/img')
DEFACTO_SP = Path('/kaggle/input/datasets/defactodataset/defactosplicing')

IMG_SIZE   = 224
BATCH_SIZE = 32
NUM_EPOCHS = 25
LR         = 1e-4

for name, p in [('CASIA_AU',CASIA_AU),('CASIA_TP',CASIA_TP),
                ('DEFACTO_CM',DEFACTO_CM),('DEFACTO_SP',DEFACTO_SP)]:
    print(f'{name:12s} exists={p.exists()}  → {p}')

In [ ]:
IMG_EXTS = {'.jpg','.jpeg','.png','.bmp','.tif','.tiff'}

def collect(folder, label, source):
    rows, folder = [], Path(folder)
    if not folder.exists():
        print(f'[MISSING] {folder}'); return rows
    for p in folder.rglob('*'):
        if p.suffix.lower() in IMG_EXTS:
            rows.append({'path': str(p), 'label': label,
                         'label_name': 'authentic' if label==0 else 'tampered',
                         'source': source})
    return rows

records  = []
records += collect(CASIA_AU,  0, 'CASIA_v2')
records += collect(CASIA_TP,  1, 'CASIA_v2')
records += collect(DEFACTO_CM, 1, 'DeFACTO_copymove')
for sp in sorted(DEFACTO_SP.glob('splicing_*_img/img')):
    records += collect(sp, 1, 'DeFACTO_splicing')

df_raw = pd.DataFrame(records)
print('── Raw counts by source ──')
print(df_raw.groupby(['source','label_name']).size(), '\n')

CAPS = {
    ('CASIA_v2', 0):         7000,
    ('CASIA_v2', 1):         None,
    ('DeFACTO_splicing', 1): 3000,
    ('DeFACTO_copymove', 1): 1800,
}

parts = []
for (src, lbl), cap in CAPS.items():
    grp = df_raw[(df_raw.source == src) & (df_raw.label == lbl)]
    n   = len(grp) if cap is None else min(cap, len(grp))
    parts.append(grp.sample(n=n, replace=False, random_state=SEED))

df = pd.concat(parts).sample(frac=1, random_state=SEED).reset_index(drop=True)

print('── After sampling ──')
print(df.groupby(['source','label_name']).size())
print(f"\nAuth: {(df.label==0).sum()} | Tamp: {(df.label==1).sum()} | Total: {len(df)}")
tamp = df[df.label==1]
print('CASIA share of tampered: '
      f"{(tamp.source=='CASIA_v2').mean():.0%}\n")

df['strat'] = df['source'] + '_' + df['label'].astype(str)
df_train, df_temp = train_test_split(df, test_size=0.25,
                                     stratify=df['strat'], random_state=SEED)
df_val, df_test   = train_test_split(df_temp, test_size=0.40,
                                     stratify=df_temp['strat'], random_state=SEED)

for nm, d in [('Train',df_train),('Val',df_val),('Test',df_test)]:
    print(f"{nm:5s} {len(d):5d} | Auth {(d.label==0).sum():4d} | Tamp {(d.label==1).sum():4d}")

In [ ]:
MEAN, STD = [0.485,0.456,0.406], [0.229,0.224,0.225]

train_tf = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.RandomRotate90(p=0.3),
    A.ImageCompression(quality_range=(60,100), p=0.4),
    A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.4),
    A.GaussNoise(noise_scale_factor=0.1, p=0.3),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])
eval_tf = A.Compose([A.Resize(IMG_SIZE,IMG_SIZE), A.Normalize(mean=MEAN,std=STD), ToTensorV2()])

class ForgeryDataset(Dataset):
    def __init__(self, frame, tf): self.df=frame.reset_index(drop=True); self.tf=tf
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        try:    img = np.array(Image.open(r['path']).convert('RGB'))
        except Exception: img = np.zeros((IMG_SIZE,IMG_SIZE,3), np.uint8)
        return self.tf(image=img)['image'], int(r['label'])

train_ds = ForgeryDataset(df_train, train_tf)
val_ds   = ForgeryDataset(df_val,   eval_tf)
test_ds  = ForgeryDataset(df_test,  eval_tf)

counts = Counter(df_train['label'])
w      = {c: 1.0/n for c,n in counts.items()}
sw     = df_train['label'].map(w).tolist()
sampler = WeightedRandomSampler(sw, num_samples=len(sw), replacement=True)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=2, pin_memory=True)

imgs, lbls = next(iter(train_loader))
print('Batch', imgs.shape, '| Auth', (lbls==0).sum().item(), 'Tamp', (lbls==1).sum().item())

In [ ]:
class CNNForgeryBranch(nn.Module):
    def __init__(self, num_classes=2, feature_dim=512, dropout=0.4):
        super().__init__()
        bb = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone = nn.Sequential(*list(bb.children())[:-1])
        for i, child in enumerate(self.backbone.children()):
            if str(i) in ['0']:  # only freeze conv1, rest is trainable
                for prm in child.parameters(): prm.requires_grad = False
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(2048,1024), nn.BatchNorm1d(1024),
            nn.ReLU(True), nn.Dropout(dropout),
            nn.Linear(1024, feature_dim), nn.BatchNorm1d(feature_dim), nn.ReLU(True))
        self.classifier = nn.Sequential(nn.Dropout(dropout), nn.Linear(feature_dim, num_classes))
    def forward(self, x, return_features=False):
        f = self.head(self.backbone(x))
        logits = self.classifier(f)
        return (f, logits) if return_features else logits

model = CNNForgeryBranch().to(DEVICE)
tot = sum(p.numel() for p in model.parameters())
tr  = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total {tot:,} | Trainable {tr:,} | Frozen {tot-tr:,}')

In [ ]:
counts = Counter(df_train['label']); total = sum(counts.values())
cw = torch.tensor([total/(2*counts[i]) for i in range(2)], dtype=torch.float32).to(DEVICE)
print(f'Class weights → Auth {cw[0]:.3f} | Tamp {cw[1]:.3f}')

criterion = nn.CrossEntropyLoss(weight=cw)
optimizer = optim.AdamW([
    {'params': model.backbone.parameters(),   'lr': LR*0.1},
    {'params': model.head.parameters(),       'lr': LR},
    {'params': model.classifier.parameters(), 'lr': LR},
], weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NUM_EPOCHS)

def run_epoch(loader, train=False):
    model.train() if train else model.eval()
    tot_loss, P, L, PR = 0.0, [], [], []
    with torch.set_grad_enabled(train):
        for imgs, lbls in loader:
            imgs, lbls = imgs.to(DEVICE), lbls.to(DEVICE)
            logits = model(imgs); loss = criterion(logits, lbls)
            if train:
                optimizer.zero_grad(); loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), 1.0); optimizer.step()
            tot_loss += loss.item()*imgs.size(0)
            PR.append(torch.softmax(logits,1)[:,1].detach().cpu())
            P.append(logits.argmax(1).cpu()); L.append(lbls.cpu())
    P,L,PR = torch.cat(P),torch.cat(L),torch.cat(PR)
    return tot_loss/len(loader.dataset), P, L, PR

history = {'train_loss':[],'val_loss':[],'train_acc':[],'val_acc':[],'val_f1':[]}
best_f1, CKPT = 0.0, '/kaggle/working/cnn_forgery_best.pth'

for ep in range(1, NUM_EPOCHS+1):
    trl, trP, trL, _ = run_epoch(train_loader, train=True)
    vll, vP,  vL,  _ = run_epoch(val_loader,   train=False)
    scheduler.step()
    tra = accuracy_score(trL, trP); vaa = accuracy_score(vL, vP)
    vf1 = f1_score(vL, vP)
    for k,v in [('train_loss',trl),('val_loss',vll),('train_acc',tra),
                ('val_acc',vaa),('val_f1',vf1)]: history[k].append(v)
    print(f'Ep {ep:02d}/{NUM_EPOCHS} | train loss {trl:.3f} acc {tra:.3f} | '
          f'val loss {vll:.3f} acc {vaa:.3f} f1 {vf1:.3f}')
    if vf1 > best_f1:
        best_f1 = vf1
        torch.save({'epoch':ep,'model_state_dict':model.state_dict(),'val_f1':best_f1}, CKPT)
        print(f'  ✓ saved (val f1 {best_f1:.3f})')

print('Best val F1:', round(best_f1,4))

In [ ]:
ck = torch.load(CKPT, map_location=DEVICE)
model.load_state_dict(ck['model_state_dict'])
print(f"Loaded epoch {ck['epoch']} (val F1 {ck['val_f1']:.3f})\n")

_, preds, labels, probs = run_epoch(test_loader, train=False)
preds, labels, probs = preds.numpy(), labels.numpy(), probs.numpy()

print('── Overall test ──')
print(f'Accuracy  {accuracy_score(labels,preds):.4f}')
print(f'Precision {precision_score(labels,preds):.4f}')
print(f'Recall    {recall_score(labels,preds):.4f}')
print(f'F1        {f1_score(labels,preds):.4f}')
print(f'AUROC     {roc_auc_score(labels,probs):.4f}\n')
print(classification_report(labels, preds, target_names=['Authentic','Tampered']))

res = df_test.reset_index(drop=True).copy()
res['pred'] = preds
res['correct'] = (res['pred'] == res['label']).astype(int)
print('── Per-source accuracy ──')
print(res.groupby('source')['correct'].mean().round(4))
print('\n── CASIA v2 only (primary benchmark) ──')
cm = res['source']=='CASIA_v2'
print(classification_report(res.loc[cm,'label'], res.loc[cm,'pred'],
                            target_names=['Authentic','Tampered']))

fig, ax = plt.subplots(1,2, figsize=(13,5))
sns.heatmap(confusion_matrix(labels,preds), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Auth','Tamp'], yticklabels=['Auth','Tamp'], ax=ax[0])
ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('Actual'); ax[0].set_title('Confusion (test)')
ax[1].plot(history['val_acc'], label='val acc', marker='o', ms=3)
ax[1].plot(history['val_f1'],  label='val f1', marker='o', ms=3, ls='--')
ax[1].plot(history['train_acc'],label='train acc', marker='o', ms=3, alpha=.6)
ax[1].set_xlabel('epoch'); ax[1].legend(); ax[1].grid(alpha=.3); ax[1].set_title('Curves')
plt.tight_layout(); plt.show()

In [ ]:
SAVE = '/kaggle/working/session_save'; os.makedirs(SAVE, exist_ok=True)
df_train.to_csv(f'{SAVE}/df_train.csv', index=False)
df_val.to_csv(f'{SAVE}/df_val.csv',     index=False)
df_test.to_csv(f'{SAVE}/df_test.csv',   index=False)
with open(f'{SAVE}/history.pkl','wb') as f: pickle.dump(history, f)
if os.path.exists(CKPT): shutil.copy(CKPT, f'{SAVE}/cnn_forgery_best.pth')
print('Saved:', os.listdir(SAVE))